# Raw Stream Arbitrage Simulation

This notebook streams raw venue `.jsonl.gz` files directly from `persist/pop-data/<run-folder>`, merges `book_state` rows across venues by `ts_book_ns`, builds only the current unified market state, and simulates cross-exchange arbitrage without materializing the full history in memory.

Primary goals:
- simulation-first architecture
- bounded memory usage
- auditability through persisted milestone, trade log, and summary outputs

This version supports:
- multiple venues
- multiple Monte Carlo paths
- latency sampling
- fee/slippage adjustments
- line-based resume milestone
- `all_at_once` execution strategy


In [2]:
from pathlib import Path

RUN_FOLDER = "2026-03-05-20-27-39"
POP_DATA_ROOT = Path("/home/fkonrad97/projects/pop/persist/pop-data")
INPUT_ROOT = POP_DATA_ROOT / RUN_FOLDER

SYMBOL_FILTER = "BTCUSDT"
MERGE_VENUES = []  # Example: ["binance", "okx"]. Empty means all discovered venues.
MAX_BOOK_AGE_MS = 2500
BOOKSTATE_BUFFER_SIZE = 256  # valid book_state rows prefetched per venue reader

SIM_CONFIG = {
    "n_paths": 16,
    "base_seed": 42,
    "strategy": "all_at_once",
    "max_trade_qty": 0.01,
    "min_net_edge_quote": 0.0,
    "fee_bps_by_venue": {
        "binance": 10.0,
        "okx": 10.0,
        "bybit": 10.0,
        "kucoin": 10.0,
    },
    "slippage_bps_by_venue": {
        "binance": 0.0,
        "okx": 0.0,
        "bybit": 0.0,
        "kucoin": 0.0,
    },
    "latency_model": {
        "kind": "uniform_ms",
        "min_ms": 50,
        "max_ms": 250,
    },
    "initial_balances": {
        "binance": {"base": 0.05, "quote": 5000.0},
        "okx": {"base": 0.05, "quote": 5000.0},
        "bybit": {"base": 0.05, "quote": 5000.0},
        "kucoin": {"base": 0.05, "quote": 5000.0},
    },
}

SAVE_RESULTS = True
CHECKPOINT_EVERY_PAGES = 5000
EQUITY_SAMPLE_EVERY_PAGES = 100

OUTPUT_ROOT = Path("/home/fkonrad97/projects/pop/persist/simulation-cache") / RUN_FOLDER
TRADES_OUTPUT_PATH = OUTPUT_ROOT / f"raw_stream_trades_{(SYMBOL_FILTER or 'all_symbols').lower()}_{RUN_FOLDER}.pkl.gz"
EQUITY_OUTPUT_PATH = OUTPUT_ROOT / f"raw_stream_equity_{(SYMBOL_FILTER or 'all_symbols').lower()}_{RUN_FOLDER}.pkl.gz"
SUMMARY_OUTPUT_PATH = OUTPUT_ROOT / f"raw_stream_summary_{(SYMBOL_FILTER or 'all_symbols').lower()}_{RUN_FOLDER}.pkl.gz"
MILESTONE_PATH = OUTPUT_ROOT / f"raw_stream_simulation_milestone_{(SYMBOL_FILTER or 'all_symbols').lower()}_{RUN_FOLDER}.pkl"


## Config Notes

- `INPUT_ROOT`: one raw run folder under `persist/pop-data/<run-folder>`.
- `MERGE_VENUES`: venue subset to include. Leave empty to use all venues found in the run folder.
- `MAX_BOOK_AGE_MS`: stale venue books older than this threshold are ignored when building the current unified state.
- `BOOKSTATE_BUFFER_SIZE`: number of valid `book_state` rows prefetched per venue reader. Larger values reduce Python read overhead but use more memory.
- `CHECKPOINT_EVERY_PAGES`: write one simulation milestone after this many processed unified timestamps.
- `EQUITY_SAMPLE_EVERY_PAGES`: do not store equity on every page unless needed. Sampling keeps output memory smaller.
- Milestone resume is line-based. On resume, each venue file is reread and skipped up to the stored line count.


In [3]:
from __future__ import annotations

import gzip
import heapq
import json
from dataclasses import asdict, dataclass, field
from collections import deque
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def normalize_symbol(symbol: str) -> str:
    """Normalize venue-specific symbol strings into one canonical comparable symbol."""
    base = str(symbol).split("@", 1)[0]
    return "".join(ch for ch in base.upper() if ch.isalnum())


def normalize_venue_name(value: str) -> str:
    """Normalize venue identifiers so file names, config values, and payload fields compare reliably."""
    return "".join(ch for ch in str(value).upper() if ch.isalnum()).lower()


def sanitize_df_for_pickle(df: pd.DataFrame) -> pd.DataFrame:
    """Convert extension dtypes into plain pandas/object dtypes before persisting outputs."""
    if df.empty:
        return df.copy()
    out = df.copy()
    out.columns = [str(col) for col in out.columns]
    for column in out.columns:
        series = out[column]
        if pd.api.types.is_string_dtype(series.dtype):
            out[column] = series.astype(object)
        elif pd.api.types.is_integer_dtype(series.dtype):
            out[column] = series.astype("int64")
        elif pd.api.types.is_float_dtype(series.dtype):
            out[column] = series.astype("float64")
    return out


def discover_raw_inputs(input_root: Path, merge_venues: list[str]) -> list[Path]:
    """Discover raw venue `.jsonl` / `.jsonl.gz` files for one run folder and optional venue subset."""
    if not input_root.exists() or not input_root.is_dir():
        return []
    allowed = {normalize_venue_name(item) for item in merge_venues} if merge_venues else None
    inputs: list[Path] = []
    for path in sorted(input_root.iterdir()):
        if not path.is_file():
            continue
        name = path.name.lower()
        if not (name.endswith(".jsonl") or name.endswith(".jsonl.gz")):
            continue
        venue = normalize_venue_name(path.stem.replace('.jsonl', ''))
        if allowed is not None and venue not in allowed:
            continue
        inputs.append(path)
    return inputs


def open_text(path: Path):
    """Open raw persisted venue files as text, including gzip-compressed files."""
    if path.name.lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return path.open("rt", encoding="utf-8")


@dataclass
class PendingOrder:
    execute_ts_ns: int
    created_ts_ns: int
    buy_venue: str
    sell_venue: str
    qty: float


@dataclass
class PathState:
    path_id: int
    rng: np.random.Generator
    balances: dict[str, dict[str, float]]
    pending_orders: list[PendingOrder] = field(default_factory=list)
    trades: list[dict[str, Any]] = field(default_factory=list)
    equity_curve: list[dict[str, Any]] = field(default_factory=list)
    diagnostics: dict[str, int] = field(default_factory=lambda: {
        "opportunities_seen": 0,
        "rejected_below_edge": 0,
        "rejected_no_qty": 0,
        "orders_scheduled": 0,
        "orders_executed": 0,
        "orders_failed_missing_book": 0,
        "orders_failed_zero_qty": 0,
    })


@dataclass
class RawVenueStream:
    """Read one raw venue file sequentially and resume from a stored line offset when needed.

    The reader keeps a small in-memory buffer of already parsed valid `book_state` rows.
    This reduces repeated `readline()` and `json.loads()` overhead while keeping memory
    bounded by `buffer_size` per venue.
    """

    venue: str
    path: Path
    symbol_filter: str | None
    line_no: int = 0
    handle: Any | None = None
    buffer_size: int = 1
    buffer: Any = field(default_factory=deque)

    def open(self) -> None:
        if self.handle is not None:
            return
        self.handle = open_text(self.path)
        skipped = 0
        while skipped < self.line_no:
            line = self.handle.readline()
            if not line:
                break
            skipped += 1
        self.line_no = skipped

    def close(self) -> None:
        if self.handle is not None:
            self.handle.close()
            self.handle = None

    def _fill_buffer(self) -> None:
        """Prefetch up to `buffer_size` valid book_state rows into the per-venue buffer."""
        self.open()
        while len(self.buffer) < int(self.buffer_size):
            line = self.handle.readline()
            if not line:
                break
            self.line_no += 1
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("event_type") != "book_state":
                continue
            venue = normalize_venue_name(obj.get("venue", self.venue))
            symbol = str(obj.get("symbol", ""))
            canonical_symbol = normalize_symbol(symbol)
            if self.symbol_filter and canonical_symbol != self.symbol_filter:
                continue
            bids = obj.get("bids", [])
            asks = obj.get("asks", [])
            if not bids or not asks:
                continue
            ts_book_ns = int(obj.get("ts_book_ns", 0))
            if ts_book_ns <= 0:
                continue
            self.buffer.append({
                "venue": venue,
                "symbol": symbol,
                "canonical_symbol": canonical_symbol,
                "ts_book_ns": ts_book_ns,
                "ts_book_dt": pd.to_datetime(ts_book_ns, unit="ns"),
                "top_n": int(obj.get("top_n", 0)),
                "source_path": str(self.path),
                "best_bid_price": float(bids[0]["price"]),
                "best_bid_qty": float(bids[0]["quantity"]),
                "best_ask_price": float(asks[0]["price"]),
                "best_ask_qty": float(asks[0]["quantity"]),
            })

    def next_book_state(self) -> dict[str, Any] | None:
        """Return the next valid `book_state` row for this venue, refilling a bounded prefetch buffer as needed."""
        if not self.buffer:
            self._fill_buffer()
        if not self.buffer:
            return None
        row = self.buffer.popleft()
        if not self.buffer:
            self._fill_buffer()
        return row


def build_initial_balances(config: dict[str, Any]) -> dict[str, dict[str, float]]:
    """Copy initial per-venue balances so each path owns its own mutable state."""
    out: dict[str, dict[str, float]] = {}
    for venue, bal in config.get("initial_balances", {}).items():
        out[normalize_venue_name(venue)] = {
            "base": float(bal.get("base", 0.0)),
            "quote": float(bal.get("quote", 0.0)),
        }
    return out


def fee_fraction(config: dict[str, Any], venue: str) -> float:
    return float(config.get("fee_bps_by_venue", {}).get(venue, 0.0)) / 10_000.0


def slippage_fraction(config: dict[str, Any], venue: str) -> float:
    return float(config.get("slippage_bps_by_venue", {}).get(venue, 0.0)) / 10_000.0


def sample_latency_ms(rng: np.random.Generator, latency_model: dict[str, Any]) -> float:
    """Sample one execution latency in milliseconds for the current simulation path."""
    kind = latency_model.get("kind", "uniform_ms")
    if kind == "uniform_ms":
        return float(rng.uniform(float(latency_model.get("min_ms", 0.0)), float(latency_model.get("max_ms", 0.0))))
    if kind == "fixed_ms":
        return float(latency_model.get("value_ms", 0.0))
    raise ValueError(f"Unsupported latency model: {kind}")


def build_page(ts_book_ns: int, latest_by_venue: dict[str, dict[str, Any]], max_book_age_ms: int) -> dict[str, Any]:
    """Build the current unified cross-venue top-of-book page from the latest fresh venue states."""
    page_ts = pd.to_datetime(ts_book_ns, unit="ns")
    fresh_books: list[dict[str, Any]] = []
    for venue, book in latest_by_venue.items():
        age_ms = (ts_book_ns - int(book["ts_book_ns"])) / 1_000_000
        if age_ms > max_book_age_ms:
            continue
        fresh_books.append({
            "venue": venue,
            "symbol": book["symbol"],
            "ts_book_ns": int(book["ts_book_ns"]),
            "ts_book_dt": book["ts_book_dt"],
            "age_ms": age_ms,
            "best_bid_price": float(book["best_bid_price"]),
            "best_bid_qty": float(book["best_bid_qty"]),
            "best_ask_price": float(book["best_ask_price"]),
            "best_ask_qty": float(book["best_ask_qty"]),
        })
    fresh_books.sort(key=lambda item: item["venue"])
    bids = sorted(fresh_books, key=lambda item: (-item["best_bid_price"], item["venue"]))
    asks = sorted(fresh_books, key=lambda item: (item["best_ask_price"], item["venue"]))
    best_bid = bids[0] if bids else None
    best_ask = asks[0] if asks else None
    spread = float(best_bid["best_bid_price"] - best_ask["best_ask_price"]) if best_bid and best_ask else None
    return {
        "ts_book_ns": ts_book_ns,
        "ts_book_dt": page_ts,
        "venue_count": int(len(fresh_books)),
        "venue_books": fresh_books,
        "best_bid": best_bid,
        "best_ask": best_ask,
        "spread": spread,
    }


def detect_opportunity(page: dict[str, Any], config: dict[str, Any]) -> dict[str, Any] | None:
    """Find the best current cross-venue top-of-book opportunity after fee/slippage adjustment."""
    venue_books = page.get("venue_books", [])
    if len(venue_books) < 2:
        return None
    bids = sorted(venue_books, key=lambda item: (-item["best_bid_price"], item["venue"]))
    asks = sorted(venue_books, key=lambda item: (item["best_ask_price"], item["venue"]))
    chosen_bid = None
    chosen_ask = None
    for ask in asks:
        for bid in bids:
            if ask["venue"] == bid["venue"]:
                continue
            chosen_ask = ask
            chosen_bid = bid
            break
        if chosen_bid is not None:
            break
    if chosen_bid is None or chosen_ask is None:
        return None
    buy_venue = str(chosen_ask["venue"])
    sell_venue = str(chosen_bid["venue"])
    buy_px = float(chosen_ask["best_ask_price"])
    sell_px = float(chosen_bid["best_bid_price"])
    raw_edge = sell_px - buy_px
    buy_cost_px = buy_px * (1.0 + fee_fraction(config, buy_venue) + slippage_fraction(config, buy_venue))
    sell_take_px = sell_px * (1.0 - fee_fraction(config, sell_venue) - slippage_fraction(config, sell_venue))
    net_edge = sell_take_px - buy_cost_px
    return {
        "ts_book_ns": int(page["ts_book_ns"]),
        "ts_book_dt": page["ts_book_dt"],
        "buy_venue": buy_venue,
        "sell_venue": sell_venue,
        "buy_price": buy_px,
        "sell_price": sell_px,
        "buy_qty": float(chosen_ask["best_ask_qty"]),
        "sell_qty": float(chosen_bid["best_bid_qty"]),
        "raw_edge_quote": raw_edge,
        "net_edge_quote": net_edge,
    }


def feasible_qty(opportunity: dict[str, Any], balances: dict[str, dict[str, float]], config: dict[str, Any]) -> float:
    """Compute feasible trade quantity using balances, top-of-book size, and configured max size."""
    buy_venue = opportunity["buy_venue"]
    sell_venue = opportunity["sell_venue"]
    buy_px = float(opportunity["buy_price"])
    max_trade_qty = float(config.get("max_trade_qty", float("inf")))
    buy_fee_slip = fee_fraction(config, buy_venue) + slippage_fraction(config, buy_venue)
    effective_buy_px = buy_px * (1.0 + buy_fee_slip)
    quote_cap = float(balances.get(buy_venue, {}).get("quote", 0.0)) / effective_buy_px if effective_buy_px > 0 else 0.0
    base_cap = float(balances.get(sell_venue, {}).get("base", 0.0))
    qty = min(max_trade_qty, float(opportunity["buy_qty"]), float(opportunity["sell_qty"]), quote_cap, base_cap)
    return max(0.0, float(qty))


def execute_due_orders(path_state: PathState, page: dict[str, Any], config: dict[str, Any]) -> None:
    """Execute any pending orders whose latency delay has elapsed by the current page timestamp."""
    current_ts_ns = int(page["ts_book_ns"])
    venue_books = {book["venue"]: book for book in page.get("venue_books", [])}
    remaining: list[PendingOrder] = []
    for order in path_state.pending_orders:
        if current_ts_ns < int(order.execute_ts_ns):
            remaining.append(order)
            continue
        buy_book = venue_books.get(order.buy_venue)
        sell_book = venue_books.get(order.sell_venue)
        if buy_book is None or sell_book is None:
            path_state.diagnostics["orders_failed_missing_book"] += 1
            continue

        buy_px = float(buy_book["best_ask_price"])
        sell_px = float(sell_book["best_bid_price"])
        qty_cap = min(float(order.qty), float(buy_book["best_ask_qty"]), float(sell_book["best_bid_qty"]))
        buy_fee = fee_fraction(config, order.buy_venue)
        sell_fee = fee_fraction(config, order.sell_venue)
        buy_slip = slippage_fraction(config, order.buy_venue)
        sell_slip = slippage_fraction(config, order.sell_venue)
        effective_buy_px = buy_px * (1.0 + buy_fee + buy_slip)
        effective_sell_px = sell_px * (1.0 - sell_fee - sell_slip)
        quote_cap = float(path_state.balances.get(order.buy_venue, {}).get("quote", 0.0)) / effective_buy_px if effective_buy_px > 0 else 0.0
        base_cap = float(path_state.balances.get(order.sell_venue, {}).get("base", 0.0))
        qty = max(0.0, min(qty_cap, quote_cap, base_cap))
        if qty <= 0:
            path_state.diagnostics["orders_failed_zero_qty"] += 1
            continue
        buy_notional = qty * effective_buy_px
        sell_notional = qty * effective_sell_px
        path_state.balances[order.buy_venue]["quote"] -= buy_notional
        path_state.balances[order.buy_venue]["base"] += qty
        path_state.balances[order.sell_venue]["base"] -= qty
        path_state.balances[order.sell_venue]["quote"] += sell_notional
        path_state.trades.append({
            "path_id": int(path_state.path_id),
            "signal_ts_ns": int(order.created_ts_ns),
            "exec_ts_ns": int(current_ts_ns),
            "exec_ts_dt": page["ts_book_dt"],
            "buy_venue": order.buy_venue,
            "sell_venue": order.sell_venue,
            "qty": float(qty),
            "buy_price": float(buy_px),
            "sell_price": float(sell_px),
            "pnl_quote": float(sell_notional - buy_notional),
        })
        path_state.diagnostics["orders_executed"] += 1
    path_state.pending_orders = remaining


def maybe_schedule_order(path_state: PathState, page: dict[str, Any], config: dict[str, Any]) -> dict[str, Any] | None:
    """Schedule one pending order from the current page if the opportunity survives config checks."""
    if config.get("strategy") != "all_at_once":
        raise ValueError(f"Unsupported strategy: {config.get('strategy')}")
    opportunity = detect_opportunity(page, config)
    if opportunity is None:
        return None
    path_state.diagnostics["opportunities_seen"] += 1
    if float(opportunity["net_edge_quote"]) < float(config.get("min_net_edge_quote", 0.0)):
        path_state.diagnostics["rejected_below_edge"] += 1
        return None
    qty = feasible_qty(opportunity, path_state.balances, config)
    if qty <= 0:
        path_state.diagnostics["rejected_no_qty"] += 1
        return None
    latency_ms = sample_latency_ms(path_state.rng, config.get("latency_model", {}))
    execute_ts_ns = int(int(page["ts_book_ns"]) + latency_ms * 1_000_000)
    path_state.pending_orders.append(PendingOrder(
        execute_ts_ns=execute_ts_ns,
        created_ts_ns=int(page["ts_book_ns"]),
        buy_venue=str(opportunity["buy_venue"]),
        sell_venue=str(opportunity["sell_venue"]),
        qty=float(qty),
    ))
    path_state.diagnostics["orders_scheduled"] += 1
    return {
        "path_id": int(path_state.path_id),
        "signal_ts_ns": int(page["ts_book_ns"]),
        "signal_ts_dt": page["ts_book_dt"],
        "buy_venue": str(opportunity["buy_venue"]),
        "sell_venue": str(opportunity["sell_venue"]),
        "qty": float(qty),
        "net_edge_quote": float(opportunity["net_edge_quote"]),
        "latency_ms": float(latency_ms),
    }


def quote_equity(balances: dict[str, dict[str, float]], page: dict[str, Any]) -> float:
    """Mark total equity to quote currency using current venue bid prices or the current global best bid."""
    venue_books = {book["venue"]: book for book in page.get("venue_books", [])}
    fallback_bid = float(page["best_bid"]["best_bid_price"]) if page.get("best_bid") else 0.0
    total = 0.0
    for venue, bal in balances.items():
        mark_bid = float(venue_books.get(venue, {}).get("best_bid_price", fallback_bid)) if fallback_bid or venue_books.get(venue) else fallback_bid
        total += float(bal.get("quote", 0.0))
        total += float(bal.get("base", 0.0)) * float(mark_bid or 0.0)
    return total


def serialize_path_state(path_state: PathState) -> dict[str, Any]:
    """Convert runtime simulation state into a pickle-friendly milestone payload."""
    return {
        "path_id": int(path_state.path_id),
        "balances": path_state.balances,
        "pending_orders": [asdict(order) for order in path_state.pending_orders],
        "trades": path_state.trades,
        "equity_curve": path_state.equity_curve,
        "diagnostics": path_state.diagnostics,
    }


def deserialize_path_state(payload: dict[str, Any], seed: int) -> PathState:
    """Restore one simulation path from milestone payload and reseed its RNG deterministically."""
    state = PathState(
        path_id=int(payload["path_id"]),
        rng=np.random.default_rng(seed),
        balances={venue: {"base": float(bal["base"]), "quote": float(bal["quote"])} for venue, bal in payload.get("balances", {}).items()},
    )
    state.pending_orders = [PendingOrder(**order) for order in payload.get("pending_orders", [])]
    state.trades = list(payload.get("trades", []))
    state.equity_curve = list(payload.get("equity_curve", []))
    state.diagnostics.update({key: int(value) for key, value in payload.get("diagnostics", {}).items()})
    return state


def save_milestone(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(payload, path)


def load_milestone(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    return pd.read_pickle(path)


def iter_unified_pages(streams: list[RawVenueStream], max_book_age_ms: int, initial_latest_by_venue: dict[str, dict[str, Any]] | None = None):
    """Merge raw venue streams by timestamp and emit one current unified page per unique timestamp."""
    heap: list[tuple[int, str, dict[str, Any], RawVenueStream]] = []
    latest_by_venue: dict[str, dict[str, Any]] = dict(initial_latest_by_venue or {})
    for stream in streams:
        row = stream.next_book_state()
        if row is None:
            continue
        heapq.heappush(heap, (int(row["ts_book_ns"]), stream.venue, row, stream))

    while heap:
        current_ts = int(heap[0][0])
        while heap and int(heap[0][0]) == current_ts:
            _, venue, row, stream = heapq.heappop(heap)
            latest_by_venue[venue] = row
            nxt = stream.next_book_state()
            if nxt is not None:
                heapq.heappush(heap, (int(nxt["ts_book_ns"]), stream.venue, nxt, stream))
        yield build_page(current_ts, latest_by_venue, max_book_age_ms), latest_by_venue


In [4]:
symbol_filter = normalize_symbol(SYMBOL_FILTER) if SYMBOL_FILTER else None
raw_input_paths = discover_raw_inputs(INPUT_ROOT, MERGE_VENUES)
raw_stream_summary = [{
    "venue": normalize_venue_name(path.stem.replace('.jsonl', '')),
    "path": str(path),
    "size_bytes": int(path.stat().st_size),
} for path in raw_input_paths]

{
    "input_root": str(INPUT_ROOT),
    "run_folder": RUN_FOLDER,
    "symbol_filter": symbol_filter,
    "merge_venues": MERGE_VENUES if MERGE_VENUES else "ALL",
    "raw_input_count": len(raw_input_paths),
    "outputs": {
        "milestone": str(MILESTONE_PATH),
        "trades": str(TRADES_OUTPUT_PATH),
        "equity": str(EQUITY_OUTPUT_PATH),
        "summary": str(SUMMARY_OUTPUT_PATH),
    },
}


{'input_root': '/home/fkonrad97/projects/pop/persist/pop-data/2026-03-05-20-27-39',
 'run_folder': '2026-03-05-20-27-39',
 'symbol_filter': 'BTCUSDT',
 'merge_venues': 'ALL',
 'raw_input_count': 4,
 'outputs': {'milestone': '/home/fkonrad97/projects/pop/persist/simulation-cache/2026-03-05-20-27-39/raw_stream_simulation_milestone_btcusdt_2026-03-05-20-27-39.pkl',
  'trades': '/home/fkonrad97/projects/pop/persist/simulation-cache/2026-03-05-20-27-39/raw_stream_trades_btcusdt_2026-03-05-20-27-39.pkl.gz',
  'equity': '/home/fkonrad97/projects/pop/persist/simulation-cache/2026-03-05-20-27-39/raw_stream_equity_btcusdt_2026-03-05-20-27-39.pkl.gz',
  'summary': '/home/fkonrad97/projects/pop/persist/simulation-cache/2026-03-05-20-27-39/raw_stream_summary_btcusdt_2026-03-05-20-27-39.pkl.gz'}}

In [5]:
pd.DataFrame(raw_stream_summary) if raw_stream_summary else "No matching raw venue files found for the selected run/venue filter."


,venue,path,size_bytes
0,binance,/home/fkonrad97/projects/pop/persist/pop-data/...,57785420
1,bybit,/home/fkonrad97/projects/pop/persist/pop-data/...,72591337
2,kucoin,/home/fkonrad97/projects/pop/persist/pop-data/...,464600668
3,okx,/home/fkonrad97/projects/pop/persist/pop-data/...,50766604


## Run Streaming Simulation

Resume semantics:
- if a milestone exists, each venue reader resumes from the stored line count
- the simulation restores `latest_by_venue`, balances, pending orders, trades, and sampled equity points
- milestone writes happen only after a fully processed page count boundary

This means resume is safe at page granularity, not byte granularity.


In [6]:
milestone = load_milestone(MILESTONE_PATH)

if milestone is None:
    streams = [
        RawVenueStream(
            venue=normalize_venue_name(path.stem.replace('.jsonl', '')),
            path=path,
            symbol_filter=symbol_filter,
            line_no=0,
            buffer_size=int(BOOKSTATE_BUFFER_SIZE),
        )
        for path in raw_input_paths
    ]
    paths = [
        PathState(
            path_id=path_id,
            rng=np.random.default_rng(int(SIM_CONFIG["base_seed"]) + path_id),
            balances=build_initial_balances(SIM_CONFIG),
        )
        for path_id in range(int(SIM_CONFIG["n_paths"]))
    ]
    resume_latest_by_venue = {}
    page_count = 0
    first_page = None
else:
    line_progress = milestone.get("line_progress", {})
    streams = [
        RawVenueStream(
            venue=normalize_venue_name(path.stem.replace('.jsonl', '')),
            path=path,
            symbol_filter=symbol_filter,
            line_no=int(line_progress.get(normalize_venue_name(path.stem.replace('.jsonl', '')), 0)),
            buffer_size=int(BOOKSTATE_BUFFER_SIZE),
        )
        for path in raw_input_paths
    ]
    paths = [
        deserialize_path_state(payload, int(SIM_CONFIG["base_seed"]) + int(payload["path_id"]))
        for payload in milestone.get("paths", [])
    ]
    resume_latest_by_venue = milestone.get("latest_by_venue", {})
    page_count = int(milestone.get("page_count", 0))
    first_page = milestone.get("first_page")

last_page = None
page_iter = iter_unified_pages(streams, MAX_BOOK_AGE_MS, initial_latest_by_venue=resume_latest_by_venue)

for page, latest_by_venue in page_iter:
    page_count += 1
    if first_page is None:
        first_page = {
            "ts_book_ns": int(page["ts_book_ns"]),
            "ts_book_dt": page["ts_book_dt"],
        }
    last_page = {
        "ts_book_ns": int(page["ts_book_ns"]),
        "ts_book_dt": page["ts_book_dt"],
    }
    for path_state in paths:
        execute_due_orders(path_state, page, SIM_CONFIG)
        maybe_schedule_order(path_state, page, SIM_CONFIG)
        if page_count % int(EQUITY_SAMPLE_EVERY_PAGES) == 0:
            path_state.equity_curve.append({
                "path_id": int(path_state.path_id),
                "ts_book_ns": int(page["ts_book_ns"]),
                "ts_book_dt": page["ts_book_dt"],
                "equity_quote": float(quote_equity(path_state.balances, page)),
                "pending_orders": int(len(path_state.pending_orders)),
            })
    if CHECKPOINT_EVERY_PAGES > 0 and page_count % int(CHECKPOINT_EVERY_PAGES) == 0:
        milestone_payload = {
            "page_count": int(page_count),
            "first_page": first_page,
            "last_page": last_page,
            "line_progress": {stream.venue: int(stream.line_no) for stream in streams},
            "latest_by_venue": latest_by_venue,
            "paths": [serialize_path_state(path_state) for path_state in paths],
        }
        save_milestone(MILESTONE_PATH, milestone_payload)

for stream in streams:
    stream.close()

trades_df = sanitize_df_for_pickle(pd.DataFrame([trade for path_state in paths for trade in path_state.trades]))
equity_df = sanitize_df_for_pickle(pd.DataFrame([point for path_state in paths for point in path_state.equity_curve]))
summary_rows = []
for path_state in paths:
    first_equity = float(path_state.equity_curve[0]["equity_quote"]) if path_state.equity_curve else 0.0
    last_equity = float(path_state.equity_curve[-1]["equity_quote"]) if path_state.equity_curve else 0.0
    summary_rows.append({
        "path_id": int(path_state.path_id),
        "trade_count": int(len(path_state.trades)),
        "final_equity_quote": last_equity,
        "pnl_quote": float(last_equity - first_equity),
        "pending_orders_left": int(len(path_state.pending_orders)),
        **{key: int(value) for key, value in path_state.diagnostics.items()},
    })
summary_df = sanitize_df_for_pickle(pd.DataFrame(summary_rows))
diagnostics_df = summary_df[[
    "path_id",
    "opportunities_seen",
    "rejected_below_edge",
    "rejected_no_qty",
    "orders_scheduled",
    "orders_executed",
    "orders_failed_missing_book",
    "orders_failed_zero_qty",
]].copy() if not summary_df.empty else pd.DataFrame()

if SAVE_RESULTS:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    trades_df.to_pickle(TRADES_OUTPUT_PATH, compression="gzip")
    equity_df.to_pickle(EQUITY_OUTPUT_PATH, compression="gzip")
    summary_df.to_pickle(SUMMARY_OUTPUT_PATH, compression="gzip")
    final_milestone = {
        "page_count": int(page_count),
        "first_page": first_page,
        "last_page": last_page,
        "line_progress": {stream.venue: int(stream.line_no) for stream in streams},
        "latest_by_venue": latest_by_venue if 'latest_by_venue' in locals() else resume_latest_by_venue,
        "paths": [serialize_path_state(path_state) for path_state in paths],
        "completed": True,
    }
    save_milestone(MILESTONE_PATH, final_milestone)

{
    "page_count": int(page_count),
    "first_page": first_page,
    "last_page": last_page,
    "trades_rows": int(len(trades_df)),
    "equity_rows": int(len(equity_df)),
    "summary_rows": int(len(summary_df)),
    "milestone_path": str(MILESTONE_PATH),
    "saved": bool(SAVE_RESULTS),
    "diagnostics_preview": diagnostics_df.to_dict(orient="records")[: min(5, len(diagnostics_df))] if not diagnostics_df.empty else [],
}


{'page_count': 5550522,
 'first_page': {'ts_book_ns': 1772738859714391888,
  'ts_book_dt': Timestamp('2026-03-05 19:27:39.714391888')},
 'last_page': None,
 'trades_rows': 0,
 'equity_rows': 888080,
 'summary_rows': 16,
 'milestone_path': '/home/fkonrad97/projects/pop/persist/simulation-cache/2026-03-05-20-27-39/raw_stream_simulation_milestone_btcusdt_2026-03-05-20-27-39.pkl',
 'saved': True,
 'diagnostics_preview': [{'path_id': 0,
   'opportunities_seen': 0,
   'rejected_below_edge': 0,
   'rejected_no_qty': 0,
   'orders_scheduled': 0,
   'orders_executed': 0,
   'orders_failed_missing_book': 0,
   'orders_failed_zero_qty': 0},
  {'path_id': 1,
   'opportunities_seen': 0,
   'rejected_below_edge': 0,
   'rejected_no_qty': 0,
   'orders_scheduled': 0,
   'orders_executed': 0,
   'orders_failed_missing_book': 0,
   'orders_failed_zero_qty': 0},
  {'path_id': 2,
   'opportunities_seen': 0,
   'rejected_below_edge': 0,
   'rejected_no_qty': 0,
   'orders_scheduled': 0,
   'orders_execute

## Outputs

- `trades_df`: executed trades with realized quote PnL
- `equity_df`: sampled equity curve by path
- `summary_df`: compact path-level outcome table
- milestone `.pkl`: resume/audit state for the run


In [7]:
summary_df.sort_values(["pnl_quote", "trade_count"], ascending=[False, False]).reset_index(drop=True) if not summary_df.empty else "No summary rows produced."


,path_id,trade_count,final_equity_quote,pnl_quote,pending_orders_left,opportunities_seen,rejected_below_edge,rejected_no_qty,orders_scheduled,orders_executed,orders_failed_missing_book,orders_failed_zero_qty
0,0,0,34245.86,18.5105,0,0,0,0,0,0,0,0
1,1,0,34245.86,18.5105,0,0,0,0,0,0,0,0
2,2,0,34245.86,18.5105,0,0,0,0,0,0,0,0
3,3,0,34245.86,18.5105,0,0,0,0,0,0,0,0
4,4,0,34245.86,18.5105,0,0,0,0,0,0,0,0
5,5,0,34245.86,18.5105,0,0,0,0,0,0,0,0
6,6,0,34245.86,18.5105,0,0,0,0,0,0,0,0
7,7,0,34245.86,18.5105,0,0,0,0,0,0,0,0
8,8,0,34245.86,18.5105,0,0,0,0,0,0,0,0
9,9,0,34245.86,18.5105,0,0,0,0,0,0,0,0


In [8]:
trades_df.head(20) if not trades_df.empty else "No trades executed."


'No trades executed.'

In [9]:
equity_df.head(20) if not equity_df.empty else "No equity points recorded."


,path_id,ts_book_ns,ts_book_dt,equity_quote,pending_orders
0,0,1772738861054805819,2026-03-05 19:27:41.054805819,34227.3495,0
1,0,1772738861151775527,2026-03-05 19:27:41.151775527,34227.3495,0
2,0,1772738861306475299,2026-03-05 19:27:41.306475299,34227.3495,0
3,0,1772738861507462423,2026-03-05 19:27:41.507462423,34227.3495,0
4,0,1772738861557284579,2026-03-05 19:27:41.557284579,34227.4345,0
5,0,1772738861734953676,2026-03-05 19:27:41.734953676,34228.5095,0
6,0,1772738861749257392,2026-03-05 19:27:41.749257392,34228.5095,0
7,0,1772738861922430785,2026-03-05 19:27:41.922430785,34228.5095,0
8,0,1772738862140755447,2026-03-05 19:27:42.140755447,34228.5045,0
9,0,1772738862198010618,2026-03-05 19:27:42.198010618,34228.3745,0


## Diagnostics

Use this section to see why a path stayed flat: whether no opportunities were seen, edges were filtered out, quantity was unavailable, or scheduled orders later failed at execution.


In [10]:
diagnostics_df.head(20) if 'diagnostics_df' in locals() and not diagnostics_df.empty else 'No diagnostics recorded.'


,path_id,opportunities_seen,rejected_below_edge,rejected_no_qty,orders_scheduled,orders_executed,orders_failed_missing_book,orders_failed_zero_qty
0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,0
5,5,0,0,0,0,0,0,0
6,6,0,0,0,0,0,0,0
7,7,0,0,0,0,0,0,0
8,8,0,0,0,0,0,0,0
9,9,0,0,0,0,0,0,0


## Load Existing Results

Use this optional section when result files already exist on disk and you want to inspect them without rerunning the full simulation.


In [11]:
LOAD_EXISTING_RESULTS = False

loaded_trades_df = pd.DataFrame()
loaded_equity_df = pd.DataFrame()
loaded_summary_df = pd.DataFrame()

if LOAD_EXISTING_RESULTS:
    loaded_trades_df = pd.read_pickle(TRADES_OUTPUT_PATH, compression="gzip") if Path(TRADES_OUTPUT_PATH).exists() else pd.DataFrame()
    loaded_equity_df = pd.read_pickle(EQUITY_OUTPUT_PATH, compression="gzip") if Path(EQUITY_OUTPUT_PATH).exists() else pd.DataFrame()
    loaded_summary_df = pd.read_pickle(SUMMARY_OUTPUT_PATH, compression="gzip") if Path(SUMMARY_OUTPUT_PATH).exists() else pd.DataFrame()
    {
        "output_root": str(OUTPUT_ROOT),
        "trades_path": str(TRADES_OUTPUT_PATH),
        "equity_path": str(EQUITY_OUTPUT_PATH),
        "summary_path": str(SUMMARY_OUTPUT_PATH),
        "trades_rows": int(len(loaded_trades_df)),
        "equity_rows": int(len(loaded_equity_df)),
        "summary_rows": int(len(loaded_summary_df)),
    }
else:
    "Set LOAD_EXISTING_RESULTS = True to load saved outputs from disk."
